In [2]:
import subprocess, shutil, os, hashlib
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

REPO    = "https://github.com/Abhinandannjain04/Realestate.git"
CLONE   = "/tmp/realestate_repo"
LANDING = "/lakehouse/default/Files/bronze/landing"
FOLDERS = ["agent","builder","customer","expense","leads","lease","property","sales"]

contracts = {
    "agent":    ["AgentID","AgentName","CommissionPercent"],
    "builder":  ["BuilderID","BuilderName","RERARegisteredFlag"],
    "customer": ["CustomerID","CustomerType","SatisfactionScore"],
    "expense":  ["ExpenseID","ExpenseAmount","ExpenseCategory"],
    "leads":    ["LeadID","LeadStatus","ConvertedToSale"],
    "lease":    ["LeaseID","MonthlyRent","LeaseTenureMonths"],
    "property": ["PropertyID","PropertyCode","BuiltUpAreaSqFt"],
    "sales":    ["SalesID","FinalSalePrice","TotalDealValue"],
}

LOG_SCHEMA = StructType([
    StructField("file_name",       StringType(),    True),
    StructField("source_folder",   StringType(),    True),
    StructField("file_sha",        StringType(),    True),
    StructField("rows_loaded",     LongType(),      True),
    StructField("processing_date", TimestampType(), True),
    StructField("status",          StringType(),    True),
    StructField("error_message",   StringType(),    True),
])

spark.sql("""CREATE TABLE IF NOT EXISTS ctl_file_log (
  file_name STRING, source_folder STRING, file_sha STRING,
  rows_loaded LONG, processing_date TIMESTAMP, status STRING, error_message STRING) USING DELTA""")

# ---- clone (with failure guard) ----
try:
    shutil.rmtree(CLONE, ignore_errors=True)
    subprocess.run(["git","clone","--depth","1",REPO,CLONE], check=True, timeout=300)
except Exception as ex:
    spark.createDataFrame(
        [("<clone>","<repo>",None,None,datetime.now(),"Failed",str(ex))], LOG_SCHEMA
    ).write.mode("append").format("delta").saveAsTable("ctl_file_log")
    mssparkutils.notebook.exit("0")

done = {r.file_sha for r in spark.sql(
        "SELECT DISTINCT file_sha FROM ctl_file_log WHERE status='Success'").collect()}

def sha_of(p):
    h = hashlib.sha256()
    with open(p,"rb") as fh:
        for c in iter(lambda: fh.read(8192), b""): h.update(c)
    return h.hexdigest()

# ---- discover, validate, load ----
log = []
for folder in FOLDERS:
    src = f"{CLONE}/{folder}"
    if not os.path.isdir(src):
        log.append(("<folder>",folder,None,None,datetime.now(),"Failed","folder not in repo")); continue
    for name in [f for f in os.listdir(src) if f.lower().endswith(".csv")]:
        local = f"{src}/{name}"; sha = sha_of(local)
        if sha in done: continue
        try:
            os.makedirs(f"{LANDING}/{folder}", exist_ok=True)
            shutil.copy(local, f"{LANDING}/{folder}/{name}")
            df = (spark.read.option("header",True).option("inferSchema",True)
                    .option("encoding","windows-1252").csv(f"Files/bronze/landing/{folder}/{name}"))
            for c in df.columns: df = df.withColumnRenamed(c, c.strip().replace(" ","_"))
            req, have = {c.lower() for c in contracts.get(folder,[])}, {c.lower() for c in df.columns}
            if req - have:
                os.remove(f"{LANDING}/{folder}/{name}")
                log.append((name,folder,sha,None,datetime.now(),"Rejected",f"missing {sorted(req-have)}")); continue
            (df.withColumn("_source_file",F.lit(name)).withColumn("_source_folder",F.lit(folder))
               .withColumn("_ingested_at",F.current_timestamp())
               .write.mode("append").option("mergeSchema",True).format("delta").saveAsTable(f"bronze_{folder}"))
            log.append((name,folder,sha,int(df.count()),datetime.now(),"Success",None))
        except Exception as ex:
            log.append((name,folder,sha,None,datetime.now(),"Failed",str(ex)))

# ---- write log with explicit schema (fixes CANNOT_DETERMINE_TYPE) ----
if log:
    spark.createDataFrame(log, LOG_SCHEMA
        ).write.mode("append").format("delta").saveAsTable("ctl_file_log")

mssparkutils.notebook.exit(str(sum(1 for x in log if x[5]=="Success")))

StatementMeta(, f9d71599-9386-4e3d-97ea-da4d216892e7, 4, Finished, Available, Finished, False)

Cloning into '/tmp/realestate_repo'...


ExitValue: 8

In [2]:
# Source = RealEstate_LH (attach RealEstate_LH as default on this notebook)
# Targets: write by full path to Silver_LH and Gold_LH

SILVER = "abfss://RealEstate_LH@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables"
GOLD   = "abfss://RealEstate_LH@onelake.dfs.fabric.microsoft.com/Gold_LH.Lakehouse/Tables"

silver_tables = ["silver_agent","silver_builder","silver_customer","silver_property",
                 "silver_sales","silver_lease","silver_leads","silver_expense"]
gold_tables   = ["gold_dim_agent","gold_dim_builder","gold_dim_customer","gold_dim_date",
                 "gold_dim_property","gold_fact_sales","gold_fact_lease",
                 "gold_kpi_agent_performance","gold_kpi_monthly_sales"]

for t in silver_tables:
    if spark.catalog.tableExists(t):
        spark.table(t).write.format("delta").mode("overwrite").save(f"{SILVER}/{t}")
        print("copied", t, "-> Silver_LH")

for t in gold_tables:
    if spark.catalog.tableExists(t):
        spark.table(t).write.format("delta").mode("overwrite").save(f"{GOLD}/{t}")
        print("copied", t, "-> Gold_LH")

print("done")

StatementMeta(, d014dd73-9cab-4f4e-97a6-a2911851cfce, 4, Finished, Available, Finished, False)

copied silver_agent -> Silver_LH
copied silver_builder -> Silver_LH
copied silver_customer -> Silver_LH
copied silver_property -> Silver_LH
copied silver_sales -> Silver_LH
copied silver_lease -> Silver_LH
copied silver_leads -> Silver_LH
copied silver_expense -> Silver_LH
copied gold_dim_agent -> Gold_LH
copied gold_dim_builder -> Gold_LH
copied gold_dim_customer -> Gold_LH
copied gold_dim_date -> Gold_LH
copied gold_dim_property -> Gold_LH
copied gold_fact_sales -> Gold_LH
copied gold_fact_lease -> Gold_LH
copied gold_kpi_agent_performance -> Gold_LH
copied gold_kpi_monthly_sales -> Gold_LH
done


In [1]:
spark.sql("DELETE FROM ctl_file_log")
print("log cleared")

StatementMeta(, cecff6da-30f1-47c0-8b2c-6b25b6f6a6d6, 3, Finished, Available, Finished, False)

log cleared


In [2]:
spark.sql("SELECT COUNT(*) AS files FROM ctl_file_log WHERE status='Success'").show()

StatementMeta(, cecff6da-30f1-47c0-8b2c-6b25b6f6a6d6, 4, Finished, Available, Finished, False)

+-----+
|files|
+-----+
|    8|
+-----+



In [3]:
spark.sql("""
  SELECT file_name, source_folder, status, rows_loaded, processing_date
  FROM ctl_file_log
  ORDER BY processing_date DESC
""").show(truncate=False)

StatementMeta(, cecff6da-30f1-47c0-8b2c-6b25b6f6a6d6, 5, Finished, Available, Finished, False)

+------------------------------------------+-------------+-------+-----------+--------------------------+
|file_name                                 |source_folder|status |rows_loaded|processing_date           |
+------------------------------------------+-------------+-------+-----------+--------------------------+
|Final Dataset of Real Estate(Sales).csv   |sales        |Success|2500       |2026-06-26 06:10:32.273231|
|Final Dataset of Real Estate(Property).csv|property     |Success|5000       |2026-06-26 06:10:28.625762|
|Final Dataset of Real Estate(Lease).csv   |lease        |Success|1800       |2026-06-26 06:10:24.693354|
|Final Dataset of Real Estate(Leads).csv   |leads        |Success|3000       |2026-06-26 06:10:21.190521|
|Final Dataset of Real Estate(Expense).csv |expense      |Success|3500       |2026-06-26 06:10:17.578535|
|Final Dataset of Real Estate(Customer).csv|customer     |Success|1000       |2026-06-26 06:10:13.603303|
|Final Dataset of Real Estate(Builder).csv |bu

In [4]:
spark.sql("""
  SELECT file_name, source_folder, status, error_message, processing_date
  FROM ctl_file_log
  WHERE status = 'Rejected'
  ORDER BY processing_date DESC
""").show(truncate=False)

StatementMeta(, cecff6da-30f1-47c0-8b2c-6b25b6f6a6d6, 6, Finished, Available, Finished, False)

+---------+-------------+------+-------------+---------------+
|file_name|source_folder|status|error_message|processing_date|
+---------+-------------+------+-------------+---------------+
+---------+-------------+------+-------------+---------------+



In [5]:
spark.sql("""
  SELECT file_name, source_folder, status, rows_loaded, processing_date
  FROM ctl_file_log
  ORDER BY processing_date DESC
""").show(truncate=False)

StatementMeta(, cecff6da-30f1-47c0-8b2c-6b25b6f6a6d6, 7, Finished, Available, Finished, False)

+------------------------------------------+-------------+-------+-----------+--------------------------+
|file_name                                 |source_folder|status |rows_loaded|processing_date           |
+------------------------------------------+-------------+-------+-----------+--------------------------+
|Final Dataset of Real Estate(Sales).csv   |sales        |Success|2500       |2026-06-26 06:10:32.273231|
|Final Dataset of Real Estate(Property).csv|property     |Success|5000       |2026-06-26 06:10:28.625762|
|Final Dataset of Real Estate(Lease).csv   |lease        |Success|1800       |2026-06-26 06:10:24.693354|
|Final Dataset of Real Estate(Leads).csv   |leads        |Success|3000       |2026-06-26 06:10:21.190521|
|Final Dataset of Real Estate(Expense).csv |expense      |Success|3500       |2026-06-26 06:10:17.578535|
|Final Dataset of Real Estate(Customer).csv|customer     |Success|1000       |2026-06-26 06:10:13.603303|
|Final Dataset of Real Estate(Builder).csv |bu

In [6]:
spark.sql("""
  SELECT file_name, source_folder, status, rows_loaded, processing_date
  FROM ctl_file_log
  ORDER BY processing_date DESC
""").show(truncate=False)

StatementMeta(, cecff6da-30f1-47c0-8b2c-6b25b6f6a6d6, 8, Finished, Available, Finished, False)

+------------------------------------------+-------------+--------+-----------+--------------------------+
|file_name                                 |source_folder|status  |rows_loaded|processing_date           |
+------------------------------------------+-------------+--------+-----------+--------------------------+
|fraud_alerts.csv                          |agent        |Rejected|NULL       |2026-06-26 06:42:19.744211|
|Final Dataset of Real Estate(Sales).csv   |sales        |Success |2500       |2026-06-26 06:10:32.273231|
|Final Dataset of Real Estate(Property).csv|property     |Success |5000       |2026-06-26 06:10:28.625762|
|Final Dataset of Real Estate(Lease).csv   |lease        |Success |1800       |2026-06-26 06:10:24.693354|
|Final Dataset of Real Estate(Leads).csv   |leads        |Success |3000       |2026-06-26 06:10:21.190521|
|Final Dataset of Real Estate(Expense).csv |expense      |Success |3500       |2026-06-26 06:10:17.578535|
|Final Dataset of Real Estate(Custome

In [1]:
df = spark.sql("SELECT count(*) FROM RealEstate_LH.dbo.bronze_agent LIMIT 1000")
display(df)

StatementMeta(, b141be91-f9ad-45bc-8d83-492961e52791, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c7b319e9-64ce-420f-82ea-74b3c47531ca)

In [4]:
spark.sql("""
  SELECT file_name, source_folder, status, rows_loaded, processing_date
  FROM ctl_file_log
  ORDER BY processing_date DESC
""").show(truncate=False)

StatementMeta(, 0fc8237f-4f69-4674-aa9a-420f6a44f7f2, 6, Finished, Available, Finished, False)

+------------------------------------------+-------------+--------+-----------+--------------------------+
|file_name                                 |source_folder|status  |rows_loaded|processing_date           |
+------------------------------------------+-------------+--------+-----------+--------------------------+
|agentdata(2).csv                          |agent        |Success |4          |2026-06-26 09:16:41.746104|
|fraud_alerts.csv                          |agent        |Rejected|NULL       |2026-06-26 09:16:27.369683|
|fraud_alerts.csv                          |agent        |Rejected|NULL       |2026-06-26 06:56:43.179546|
|fraud_alerts.csv                          |agent        |Rejected|NULL       |2026-06-26 06:42:19.744211|
|Final Dataset of Real Estate(Sales).csv   |sales        |Success |2500       |2026-06-26 06:10:32.273231|
|Final Dataset of Real Estate(Property).csv|property     |Success |5000       |2026-06-26 06:10:28.625762|
|Final Dataset of Real Estate(Lease).

In [5]:
df = spark.sql("SELECT count (*) FROM RealEstate_LH.dbo.bronze_agent LIMIT 1000")
display(df)

StatementMeta(, 0fc8237f-4f69-4674-aa9a-420f6a44f7f2, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 73d39718-9d60-406d-b856-3c9430d782b4)

In [2]:
spark.sql("""
  SELECT file_name, source_folder, status, rows_loaded, processing_date
  FROM ctl_file_log
  ORDER BY processing_date DESC
""").show(truncate=False)

StatementMeta(, b141be91-f9ad-45bc-8d83-492961e52791, 4, Finished, Available, Finished, False)

+------------------------------------------+-------------+--------+-----------+--------------------------+
|file_name                                 |source_folder|status  |rows_loaded|processing_date           |
+------------------------------------------+-------------+--------+-----------+--------------------------+
|agents(2).csv                             |agent        |Rejected|NULL       |2026-06-26 12:21:45.050066|
|fraud_alerts.csv                          |agent        |Rejected|NULL       |2026-06-26 12:21:44.015663|
|fraud_alerts.csv                          |agent        |Rejected|NULL       |2026-06-26 10:56:57.863315|
|agentdata(2).csv                          |agent        |Success |4          |2026-06-26 09:16:41.746104|
|fraud_alerts.csv                          |agent        |Rejected|NULL       |2026-06-26 09:16:27.369683|
|fraud_alerts.csv                          |agent        |Rejected|NULL       |2026-06-26 06:56:43.179546|
|fraud_alerts.csv                    